# Raw Dataset Processor

Pipeline monitoring untuk mengubah raw dataset `datasets/url_discovery/*.json` menjadi kandidat dataset dengan schema seperti `datasets/v1_shs_datasets.csv`. Notebook ini tidak menyimpan output secara otomatis.

## 1. Setup

In [122]:
from pathlib import Path
import sys

import polars as pl
from IPython.display import display


def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in (current, *current.parents):
        if (candidate / "config.py").exists() and (candidate / "services").is_dir():
            return candidate
    raise FileNotFoundError("Root proyek tidak ditemukan")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import config
from services.dataset_service import DatasetService

pl.Config.set_tbl_hide_column_data_types(True)
pl.Config.set_tbl_cell_alignment("LEFT")
pl.Config.set_fmt_str_lengths(250)
pl.Config.set_tbl_width_chars(180)
pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_rows(20)

dataset_service = DatasetService()
RAW_FOLDER = config.DATASETS / "url_discovery"
RESEARCH_CONFIG_PATH = PROJECT_ROOT / "research_config.json"
REFERENCE_DATASET_PATH = config.DATASETS / "v1_shs_datasets.csv"

print(f"Raw folder: {RAW_FOLDER}")
print(f"Research config: {RESEARCH_CONFIG_PATH}")
print(f"Reference schema: {REFERENCE_DATASET_PATH}")

Raw folder: E:\School\tugas-akhir\project\datasets\url_discovery
Research config: E:\School\tugas-akhir\project\research_config.json
Reference schema: E:\School\tugas-akhir\project\datasets\v1_shs_datasets.csv


## 2. Load Config dan Raw Discovery

In [123]:
research_config = dataset_service.load_research_config(RESEARCH_CONFIG_PATH)
meta_df = dataset_service.load_url_discovery_meta(RAW_FOLDER)
queries_df = dataset_service.load_url_discovery_queries(RAW_FOLDER)
raw_records_df = dataset_service.load_url_discovery_records(RAW_FOLDER)

RECORDS_FILTER_CONDITIONS = [
    # pl.col("content_status") == "success",
    # pl.col("content_text").fill_null("").str.strip_chars().str.len_chars() > 0,
    # ~pl.col("domain").str.contains("instagram.com|tiktok.com|facebook.com"),
]


def apply_filter_conditions(df: pl.DataFrame, conditions: list) -> pl.DataFrame:
    if not conditions:
        return df
    expression = conditions[0]
    for condition in conditions[1:]:
        expression = expression & condition
    return df.filter(expression)


records_df = apply_filter_conditions(raw_records_df, RECORDS_FILTER_CONDITIONS)

print("Search label:", research_config.get("search_label"))
print("Primary location:", research_config.get("primary_location"))
print(f"File metadata: {meta_df.height:,}")
print(f"Query rows: {queries_df.height:,}")
print(f"Raw records unik: {raw_records_df.height:,}")
print(f"Records setelah filter manual: {records_df.height:,}")

display(meta_df)

Search label: SHS/PLTS Kalimantan Barat
Primary location: Kalimantan Barat
File metadata: 6
Query rows: 1,310
Raw records unik: 855
Records setelah filter manual: 855


source_file,built_at,search_label,source_files,record_count,content_count,query_count
"""dataset_20260627T094855Z.json""","""2026-06-27T09:48:55.240893+00:00""","""SHS/PLTS Kalimantan Barat""","{""E:\Work\personal\url-discovery\data\urls.json"",""E:\Work\personal\url-discovery\data\contents.json"",""E:\Work\personal\url-discovery\data\search_state.json""}",214,214,335
"""dataset_20260627T101253Z.json""","""2026-06-27T10:12:53.267116+00:00""","""SHS/PLTS Kalimantan Barat""","{""E:\Work\personal\url-discovery\data\urls.json"",""E:\Work\personal\url-discovery\data\contents.json"",""E:\Work\personal\url-discovery\data\search_state.json""}",203,203,268
"""dataset_20260627T135259Z.json""","""2026-06-27T13:52:59.725144+00:00""","""SHS/PLTS Kalimantan Barat""","{""E:\Work\personal\url-discovery\data\urls.json"",""E:\Work\personal\url-discovery\data\contents.json"",""E:\Work\personal\url-discovery\data\search_state.json""}",88,88,197
"""dataset_20260628T033631Z.json""","""2026-06-28T03:36:31.219965+00:00""","""SHS/PLTS Kalimantan Barat""","{""E:\Work\personal\url-discovery\data\urls.json"",""E:\Work\personal\url-discovery\data\contents.json"",""E:\Work\personal\url-discovery\data\search_state.json""}",202,202,381
"""dataset_20260628T104512Z.json""","""2026-06-28T10:45:12.810124+00:00""","""SHS/PLTS Kalimantan Barat""","{""E:\Work\personal\url-discovery\data\urls.json"",""E:\Work\personal\url-discovery\data\contents.json"",""E:\Work\personal\url-discovery\data\search_state.json""}",136,136,369
"""dataset_20260628T113525Z.json""","""2026-06-28T11:35:25.172805+00:00""","""SHS/PLTS Kalimantan Barat""","{""E:\Work\personal\url-discovery\data\urls.json"",""E:\Work\personal\url-discovery\data\contents.json"",""E:\Work\personal\url-discovery\data\search_state.json""}",12,12,28


## 3. Monitoring Kualitas Raw Dataset

In [124]:
status_distribution = (
    records_df.with_columns(pl.col("content_status").fill_null("missing"))
    .group_by("content_status")
    .agg(pl.len().alias("jumlah"))
    .sort("jumlah", descending=True)
)

domain_distribution = (
    records_df.with_columns(pl.col("domain").fill_null("unknown"))
    .group_by("domain")
    .agg(pl.len().alias("jumlah"))
    .sort("jumlah", descending=True)
    .head(20)
)

query_group_distribution = (
    records_df.with_columns(pl.col("query_group").fill_null("unknown"))
    .group_by("query_group")
    .agg(pl.len().alias("jumlah"))
    .sort("jumlah", descending=True)
    .head(20)
)

text_length_summary = records_df.select(
    pl.len().alias("total_records"),
    (
        pl.col("content_text")
        .fill_null("")
        .str.strip_chars()
        .str.len_chars()
        > 0
    ).sum().alias("records_with_text"),
    pl.col("content_text_length").min().alias("min_text_length"),
    pl.col("content_text_length").mean().round(2).alias("avg_text_length"),
    pl.col("content_text_length").max().alias("max_text_length"),
)

display(status_distribution)
display(domain_distribution)
display(query_group_distribution)
display(text_length_summary)

content_status,jumlah
"""success""",645
"""too_short""",124
"""pdf_skipped""",45
"""failed""",41


domain,jumlah
"""www.instagram.com""",340
"""www.tiktok.com""",123
"""www.facebook.com""",25
"""rri.co.id""",21
"""kalbar.antaranews.com""",20
"""pontianakpost.jawapos.com""",19
"""www.researchgate.net""",9
"""id.scribd.com""",8
"""hijau.bisnis.com""",8
"""gatrik.esdm.go.id""",8


query_group,jumlah
"""issue:benefit""",93
"""issue:problem""",84
"""social""",75
"""issue:access""",62
"""issue:environment""",55
"""issue:experience""",50
"""entity-opinion""",48
"""issue:socioeconomic""",47
"""issue:funding""",46
"""site:government_program""",39


total_records,records_with_text,min_text_length,avg_text_length,max_text_length
855,665,0,3076.92,33760


## 4. Bangun Kandidat Schema v1

In [ ]:
candidate_df = dataset_service.build_v1_candidate_rows(
    records_df=records_df,
    research_config=research_config,
    max_text_chars=260,
)

CANDIDATE_FILTER_CONDITIONS = [
    pl.col("location") == "",
    pl.col("source_url").str.to_lowercase().str.contains("facebook.com"),
    # ~pl.col("text").str.to_lowercase().str.contains("penayangan"),
    # pl.col("aspect").is_in(["experience", "benefit", "access"]),
]

filtered_candidate_df = apply_filter_conditions(candidate_df, CANDIDATE_FILTER_CONDITIONS)

reference_df = dataset_service.load(REFERENCE_DATASET_PATH)
expected_columns = list(dataset_service.v1_dataset_columns())
missing_candidate_columns = [column for column in expected_columns if column not in filtered_candidate_df.columns]
extra_candidate_columns = [column for column in filtered_candidate_df.columns if column not in expected_columns]

print(f"Candidate rows sebelum filter kandidat: {candidate_df.height:,}")
print(f"Candidate rows setelah filter kandidat: {filtered_candidate_df.height:,}")
print("Kolom wajib hilang:", missing_candidate_columns)
print("Kolom monitoring tambahan:", extra_candidate_columns)
print("Urutan kolom utama sama:", filtered_candidate_df.columns[: len(expected_columns)] == expected_columns)

display(filtered_candidate_df.select(expected_columns).head(20))

Candidate rows sebelum filter kandidat: 855
Candidate rows setelah filter kandidat: 18
Kolom wajib hilang: []
Kolom monitoring tambahan: ['raw_source_file', 'raw_domain', 'content_status', 'query_group', 'query', 'raw_title', 'raw_text_length']
Urutan kolom utama sama: True


text_id,text,subjectivity_type,speaker_type,public_opinion_scope,corpus_role,aspect,location,sentiment_label,label_status,source_id,source_type,source_url,dataset_tier,inclusion_status,verification_status,evidence_support_score,parent_text_id,decision_note
"""RAW-0091""","""Drag the slider to fit the puzzle20260627174611F521240308DA9E98150F""","""contextual_source""","""community_representative""","""contextual_reference""","""excluded""","""general_shs""","""""","""""","""unlabeled""","""RAW-SRC-0091""","""online_news""","""https://www.tiktok.com/@atonergi/photo/7548649863374753032""","""C_holdout_excluded""","""held_out_not_for_sentiment_core""","""perlu_verifikasi""",0.1,"""""","""content_not_success; keyword_relevance_needs_review"""
"""RAW-0165""","""Drag the slider to fit the puzzle202606271747180CDC67C590DBD7C1EF77""","""contextual_source""","""community_representative""","""contextual_reference""","""excluded""","""general_shs""","""""","""""","""unlabeled""","""RAW-SRC-0165""","""online_news""","""https://www.tiktok.com/@wartabromo/photo/7598712131042774280""","""C_holdout_excluded""","""held_out_not_for_sentiment_core""","""perlu_verifikasi""",0.1,"""""","""content_not_success; keyword_relevance_needs_review"""
"""RAW-0189""","""Trascina il cursore per completare il puzzle202606271747476E725D27450ED99E4C30""","""contextual_source""","""community_representative""","""contextual_reference""","""excluded""","""general_shs""","""""","""""","""unlabeled""","""RAW-SRC-0189""","""online_news""","""https://www.tiktok.com/@ossy_dermawan/photo/7536940980524600584?image_index=2&lang=it-IT""","""C_holdout_excluded""","""held_out_not_for_sentiment_core""","""perlu_verifikasi""",0.1,"""""","""content_not_success; keyword_relevance_needs_review"""
"""RAW-0235""","""Drag the slider to fit the puzzle""","""contextual_source""","""community_representative""","""contextual_reference""","""excluded""","""general_shs""","""""","""""","""unlabeled""","""RAW-SRC-0235""","""online_news""","""https://www.tiktok.com/@atonergi/photo/7565748778045541650?image_index=2""","""C_holdout_excluded""","""held_out_not_for_sentiment_core""","""perlu_verifikasi""",0.1,"""""","""content_not_success; keyword_relevance_needs_review"""
"""RAW-0266""","""Drag the slider to fit the puzzle20260627180916B171DE670A9652C57739""","""contextual_source""","""community_representative""","""contextual_reference""","""excluded""","""general_shs""","""""","""""","""unlabeled""","""RAW-SRC-0266""","""online_news""","""https://www.tiktok.com/@melda.news.online/photo/7623327466844704020""","""C_holdout_excluded""","""held_out_not_for_sentiment_core""","""perlu_verifikasi""",0.1,"""""","""content_not_success; keyword_relevance_needs_review"""
"""RAW-0325""","""TikTokTikTok© 2026 TikTokCapCut · Captions for chat videosmfajar444 · 5-31Mau bahas mitos dan fakta apa lagi tentang PLTS? Tulis di kolom komentar ya! 👇⚡Malam ini bulan bersinar terang. Ternyata panel surya tetap merespons cahaya bulan dan menghasilk…","""public_experience""","""community_representative""","""public_opinion""","""core_public_opinion""","""general_shs""","""""","""""","""unlabeled""","""RAW-SRC-0325""","""online_news""","""https://www.tiktok.com/@mfajar444/video/7646066889839430920""","""A_candidate_core""","""candidate_analysis_ready""","""perlu_verifikasi""",0.8,"""""","""candidate_from_raw_discovery"""
"""RAW-0371""","""Access to www.tiktok.com was denied You don't have authorization to view this page. HTTP ERROR 403 You don't have authorization to view this page.""","""contextual_source""","""community_representative""","""contextual_reference""","""excluded""","""general_shs""","""""","""""","""unlabeled""","""RAW-SRC-0371""","""online_news""","""https://www.tiktok.com/@riyan_saputra28/video/7591954317863570709""","""C_holdout_excluded""","""held_out_not_for_sentiment_core""","""perlu_verifikasi""",0.2,"""""","""content_not_success; keyword_relevance_needs_review"""
"""RAW-0583""","""Drag the slide

## 5. Monitoring Kandidat

In [126]:
tier_distribution = (
    filtered_candidate_df.group_by("dataset_tier")
    .agg(pl.len().alias("jumlah"))
    .sort("jumlah", descending=True)
)

inclusion_distribution = (
    filtered_candidate_df.group_by("inclusion_status")
    .agg(pl.len().alias("jumlah"))
    .sort("jumlah", descending=True)
)

source_type_distribution = (
    filtered_candidate_df.group_by("source_type")
    .agg(pl.len().alias("jumlah"))
    .sort("jumlah", descending=True)
)

aspect_distribution = (
    filtered_candidate_df.group_by("aspect")
    .agg(pl.len().alias("jumlah"))
    .sort("jumlah", descending=True)
    .head(20)
)


location_distribution = (
    filtered_candidate_df.group_by("location")
    .agg(pl.len().alias("jumlah"))
    .sort("jumlah", descending=True)
    .head(20)
)

display(tier_distribution)
display(inclusion_distribution)
display(source_type_distribution)
display(aspect_distribution)
display(location_distribution)

dataset_tier,jumlah
"""C_holdout_excluded""",12
"""A_candidate_core""",5
"""B_review_queue""",1


inclusion_status,jumlah
"""held_out_not_for_sentiment_core""",12
"""candidate_analysis_ready""",5
"""review_required_before_core""",1


source_type,jumlah
"""online_news""",18


aspect,jumlah
"""general_shs""",16
"""experience""",2


location,jumlah
"""""",18


## 6. Review Kandidat Prioritas

In [127]:
review_columns = [
    "text_id",
    "dataset_tier",
    "inclusion_status",
    "evidence_support_score",
    "aspect",
    "location",
    "source_type",
    "source_url",
    "text",
    "decision_note",
    "query_group",
    "query",
]

priority_df = (
    filtered_candidate_df
    .filter(pl.col("dataset_tier").is_in(["A_candidate_core", "B_review_queue"]))
    .sort(["dataset_tier", "evidence_support_score"], descending=[False, True])
    .select(review_columns)
)

display(priority_df.head(50))

text_id,dataset_tier,inclusion_status,evidence_support_score,aspect,location,source_type,source_url,text,decision_note,query_group,query
"""RAW-0325""","""A_candidate_core""","""candidate_analysis_ready""",0.8,"""general_shs""","""""","""online_news""","""https://www.tiktok.com/@mfajar444/video/7646066889839430920""","""TikTokTikTok© 2026 TikTokCapCut · Captions for chat videosmfajar444 · 5-31Mau bahas mitos dan fakta apa lagi tentang PLTS? Tulis di kolom komentar ya! 👇⚡Malam ini bulan bersinar terang. Ternyata panel surya tetap merespons cahaya bulan dan menghasilk…","""candidate_from_raw_discovery""","""issue:access""","""""PLTS"" ""tanpa listrik PLN"" ""Sanggau"" after:2024-01-01 before:2026-06-27"""
"""RAW-0625""","""A_candidate_core""","""candidate_analysis_ready""",0.8,"""experience""","""""","""online_news""","""https://www.tiktok.com/@kompascom/video/7654501879640116500""","""TikTokTikTok© 2026 TikTokkompascom · 4d ago Kepastian pasokan listrik kini menjadi perhatian warga di sejumlah desa di Demak. PLN memastikan pemadaman bergilir akibat kendala teknis pembangkit tidak akan terulang. Sebelumnya, sejumlah desa sempat men…","""candidate_from_raw_discovery""","""entity-opinion""","""""listrik desa"" ""tanggapan masyarakat"" ""Desa Bunut Hulu"" after:2024-01-01 before:2026-06-27"""
"""RAW-0633""","""A_candidate_core""","""candidate_analysis_ready""",0.8,"""experience""","""""","""online_news""","""https://www.tiktok.com/@doktor.barwanto/video/7583570972150615317""","""TikTokTikTok© 2026 TikTokKarangan Hilir · East Kutai Regencydoktor.barwanto · 2025-12-14Pemerintah Provinsi Kalimantan Timur terus memperluas akses energi listrik hingga ke wilayah terpencil. Program ini kini menjangkau 120 kepala keluarga di Desa Ka…","""candidate_from_raw_discovery""","""social""","""site:tiktok.com ""PLTS daerah terpencil"" ""Kalbar"" after:2024-01-01 before:2026-06-27"""
"""RAW-0659""","""A_candidate_core""","""candidate_analysis_ready""",0.8,"""general_shs""","""""","""online_news""","""https://www.tiktok.com/@ilhambachtiarr/video/7477121483966401797""","""TikTokTikTok© 2026 TikTokBandungilhambachtiarr · 2025-3-2Berapa sih total biaya bangun PLTS 24 Jam Off Grid kaya dirumah pakcanggih? pakcanggih mau langsung totalin ya -Solar Panel 4400WP (8 lembar) : 15jt-Inverter 6000Watt Hybrid Low Freq : 8.5jt-Ba…","""candidate_from_raw_discovery""","""issue:access""","""""Solar Home System"" ""tanpa listrik PLN"" ""Sintang"" after:2024-01-01 before:2026-06-27"""
"""RAW-0740""","""A_candidate_core""","""candidate_analysis_ready""",0.8,"""general_shs""","""""","""online_news""","""https://www.tiktok.com/@drone.hiber/video/7511188909380947256""","""TikTokTikTok© 2026 TikTokCiamisdrone.hiber · 2025-6-2Plta Mini Gaes #fypp #plta #listrikgratis #listrik #kincirair #turbine #pedesaan #pln #kdm #kangdedimulyadi #gubernurjawabarat 00:00 / 00:20192346252215""","""candidate_from_raw_discovery""","""issue:problem""","""""panel surya"" ""mangkrak"" ""Sukadana"" after:2024-01-01 before:2026-06-27"""
"""RAW-0717""","""B_review_queue""","""review_required_before_core""",0.55,"""general_shs""","""""","""online_news""","""https://www.tiktok.com/@racunfashionist3/video/7579046676434898194""","""TikTokTikTok© 2026 TikTokVideo currently unavailableLooking for videos? Try browsing our trending creators, hashtags, and sounds. CaptionsShow captions (unavailable)Captions currently not supportedAlways translate postsDescriptions and captions will …","""keyword_relevance_needs_review""","""issue:access""","""""energi terbarukan"" ""tanpa listrik PLN"" ""Ketapang"" after:2024-01-01 before:2026-06-27"""


## 7. Export Manual Opsional

In [128]:
EXPORT_CSV = False
EXPORT_PATH = config.OUTPUTS / "datasets" / "raw_candidate_v1_schema.csv"

if EXPORT_CSV:
    EXPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
    filtered_candidate_df.write_csv(EXPORT_PATH)
    print(f"Candidate dataset terfilter disimpan ke: {EXPORT_PATH}")
else:
    print("Export dimatikan. Ubah EXPORT_CSV = True jika ingin menyimpan kandidat terfilter secara manual.")

Export dimatikan. Ubah EXPORT_CSV = True jika ingin menyimpan kandidat terfilter secara manual.
